# Caso integrador · El informe de cierre de año

*Análisis Avanzado de Datos con Python · Subsecretaría de Energía · Módulo 7*

Trabaja sobre tu propia copia del notebook. Todo lo que escribas queda en ella.

In [ ]:
#@title El encargo { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">El encargo</div><p>Es enero de 2026. Llega el archivo del año 2025 y la jefatura pide un informe de una página, para el jueves, con tres respuestas.</p>
<ol>
<li><strong>Cómo se comportó la demanda en 2025 respecto de 2024.</strong> No basta el total del sistema, quieren saber si el crecimiento fue parejo.</li>
<li><strong>Qué cambió en la generación.</strong> Si alguna tecnología se movió fuerte y por qué.</li>
<li><strong>Si hay algo en la bitácora de mantenimiento que valga la pena mirar.</strong></li>
</ol>
<p>Y una condición que no está escrita en el correo pero se da por supuesta. <strong>Todo lo que afirmes tiene que poder rehacerse.</strong> Si alguien pregunta de dónde salió un número, la respuesta es una celda de este notebook.</p>
<p>Tienes dos horas. No hay material nuevo, todo lo que necesitas ya lo usaste en los seis módulos.</p></div>"""))

In [ ]:
#@title Datos del curso { display-mode: "form" }
#@markdown Corre esta celda. Deja listos los archivos del Observatorio de Datos Energeticos.
import numpy as np, pandas as pd, os, json, sqlite3
if not os.path.exists("demanda_2025.csv"):
    rng = np.random.default_rng(2026)
    centrales = pd.DataFrame([
     ("Central Rio Manso Alto","hidro","Biobio",420,2004),("Central Salto Verde","hidro","Los Lagos",310,1998),
     ("Central Aguas Claras","hidro","Biobio",180,2011),("Central Vega Azul","hidro","Los Lagos",95,2016),
     ("Central Tres Saltos","hidro","Biobio",260,1995),
     ("Parque Solar Pampa Alta","solar","Antofagasta",230,2019),("Parque Solar Llano Seco","solar","Atacama",180,2020),
     ("Parque Solar Sol Naciente","solar","Antofagasta",145,2021),("Parque Solar Quebrada Honda","solar","Atacama",95,2022),
     ("Parque Solar Altiplano","solar","Antofagasta",310,2023),
     ("Eolica Cerro Negro","eolica","Coquimbo",160,2017),("Eolica Punta Ventosa","eolica","Coquimbo",120,2018),
     ("Eolica Loma Fria","eolica","Valparaiso",85,2020),("Eolica Campo Abierto","eolica","Coquimbo",200,2021),
     ("Termoelectrica Bahia Norte","gas","Valparaiso",375,2008),("Termoelectrica Puerto Sur","gas","Biobio",290,2012),
     ("Termoelectrica Valle Central","gas","Metropolitana",210,2006),
     ("Carboelectrica Costa Brava","carbon","Biobio",480,2001),("Carboelectrica Roca Gris","carbon","Antofagasta",350,1999),
     ("Diesel Respaldo Cordillera","diesel","Metropolitana",45,2014),
    ], columns=["central","tecnologia","region","potencia_mw","anio_inicio"])
    fechas = pd.date_range("2024-01-01","2024-12-31",freq="D")
    perfil = np.array([0,0,0,0,0,0,.05,.18,.38,.58,.75,.87,.93,.9,.8,.63,.42,.2,.05,0,0,0,0,0])
    filas=[]
    for _,c in centrales.iterrows():
        p,t = c.potencia_mw, c.tecnologia
        for f in fechas:
            est = 1+0.25*np.cos(2*np.pi*(f.dayofyear-15)/365)
            if t=="solar": base = p*perfil*0.30*est*rng.uniform(.8,1.1)
            elif t=="eolica": base = p*0.36*rng.uniform(.15,1.6,24)
            elif t=="hidro": base = p*0.55*(2-est)*rng.uniform(.9,1.1,24)
            elif t=="gas": base = p*0.68*rng.uniform(.9,1.05,24)
            elif t=="carbon": base = p*0.65*rng.uniform(.95,1.02,24)
            else:
                base = np.zeros(24); base[18:23] = p*0.55*rng.uniform(.8,1,5)
            filas.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                       "central":c.central,"mwh":np.clip(base,0,p).round(2)}))
    centrales.to_csv("centrales.csv", index=False)
    pd.concat(filas, ignore_index=True).to_csv("generacion.csv", index=False)

    # Excel con dos hojas, la segunda con notas en texto libre
    with pd.ExcelWriter("centrales.xlsx") as w:
        centrales.to_excel(w, sheet_name="centrales", index=False)
        pd.DataFrame({"nota":["Potencias declaradas al 31 de diciembre de 2024",
                              "Las centrales de pasada se informan con su potencia maxima"]}
                     ).to_excel(w, sheet_name="notas", index=False)

    # Demanda por región, base de datos SQLite
    regs = ["Antofagasta","Atacama","Coquimbo","Valparaiso","Metropolitana","Biobio","Los Lagos"]
    pobl = [700000,320000,850000,1900000,8100000,1700000,900000]
    perfil_d = np.array([.72,.68,.66,.65,.66,.70,.78,.88,.95,.98,1.0,1.02,1.03,1.0,.97,.96,.97,1.0,1.06,1.10,1.08,.98,.88,.79])
    dem=[]
    for r,p in zip(regs,pobl):
        base_r = p/8000
        for f in fechas:
            inv = 1+0.18*np.cos(2*np.pi*(f.dayofyear-190)/365)
            finde = 0.92 if f.dayofweek>=5 else 1.0
            v = base_r*perfil_d*inv*finde*rng.uniform(.97,1.03,24)
            dem.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                     "region":r,"mwh":v.round(2)}))
    demanda = pd.concat(dem, ignore_index=True)
    demanda.to_csv("demanda.csv", index=False)
    con = sqlite3.connect("demanda.db")
    demanda.to_sql("demanda", con, index=False, if_exists="replace")
    pd.DataFrame({"region":regs,"poblacion":pobl}).to_sql("regiones", con, index=False, if_exists="replace")
    con.close()

    # Precios de nudo de enero, como los entregaría una API REST
    ene = demanda[demanda["fecha"].str.startswith("2024-01")]
    pr = ene.assign(precio_usd_mwh=(40 + ene["mwh"]/ene["mwh"].max()*110
                                    + rng.normal(0,6,len(ene))).clip(40,180).round(2))
    with open("precios_nudo.json","w") as f:
        json.dump({"metadata":{"fuente":"Observatorio de Datos Energeticos",
                               "fecha_consulta":"2024-02-01","unidad":"USD por MWh"},
                   "datos": pr[["fecha","hora","region","precio_usd_mwh"]].to_dict("records")},
                  f)

    # ------------------------------------------------ bitacora de mantenimiento
    # Seiscientos eventos del año, con dos patrones plantados a propósito que el
    # lab va a tener que encontrar y medir.
    rngm = np.random.default_rng(404)
    TIPOS = ["preventivo","correctivo","falla_electrica","falla_mecanica",
             "evento_climatico","inspeccion"]
    PESOS = {"hidro":[.34,.14,.12,.16,.02,.22], "solar":[.38,.12,.16,.08,.02,.24],
             "eolica":[.26,.12,.12,.20,.08,.22], "gas":[.30,.18,.14,.18,.01,.19],
             "carbon":[.28,.18,.14,.20,.01,.19], "diesel":[.14,.52,.10,.12,.01,.11]}
    DURA = {"preventivo":(4,24), "correctivo":(6,72), "falla_electrica":(2,48),
            "falla_mecanica":(8,96), "evento_climatico":(3,36), "inspeccion":(1,8)}
    n_dias = len(fechas)
    invierno = np.isin(fechas.month.to_numpy(), [5,6,7,8])
    peso_clima = np.where(invierno, 4.0, 1.0); peso_clima /= peso_clima.sum()

    ev = []
    for _,c in centrales.iterrows():
        p = np.array(PESOS[c.tecnologia]); p = p/p.sum()
        n = 30 if c.tecnologia=="diesel" else 22
        for tipo, d in zip(rngm.choice(TIPOS, size=n, p=p), rngm.integers(0, n_dias, size=n)):
            ev.append([c.central, int(d), str(tipo)])
        n_cl = 12 if c.tecnologia=="eolica" else 1
        for d in rngm.choice(n_dias, size=n_cl, replace=False, p=peso_clima):
            ev.append([c.central, int(d), "evento_climatico"])

    # Patron 1, el 70 por ciento de las fallas electricas arrastra un correctivo
    # en la misma central dentro de los tres dias siguientes.
    elec = [i for i,e in enumerate(ev) if e[2]=="falla_electrica"]
    for i in sorted(rngm.choice(elec, size=int(round(len(elec)*0.70)), replace=False)):
        ev.append([ev[i][0], min(ev[i][1] + int(rngm.integers(0,4)), n_dias-1), "correctivo"])
    # Patron 2, la mitad de los eventos climaticos trae una falla mecanica el mismo dia.
    clim = [i for i,e in enumerate(ev) if e[2]=="evento_climatico"]
    for i in sorted(rngm.choice(clim, size=int(round(len(clim)*0.50)), replace=False)):
        ev.append([ev[i][0], ev[i][1], "falla_mecanica"])

    nombres = centrales["central"].to_numpy()
    while len(ev) < 600:
        ev.append([str(nombres[int(rngm.integers(0,len(nombres)))]),
                   int(rngm.integers(0,n_dias)),
                   "preventivo" if rngm.random()<0.6 else "inspeccion"])
    ev = ev[:600]

    mant = pd.DataFrame([{"central":c, "fecha":fechas[d].strftime("%Y-%m-%d"),
                          "tipo_evento":t,
                          "duracion_horas":int(rngm.integers(DURA[t][0], DURA[t][1]+1))}
                         for c,d,t in ev])

    # ------------------------------------------- texto libre de cada evento
    # Una observación escrita como la escribiría el turno, con vocabulario
    # propio de cada tipo. El Módulo 5 busca temas ahí adentro.
    VOCAB = {
     "falla_electrica": (["el transformador de poder","el interruptor principal",
        "la barra de media tensión","el relé de protección","el aislador de línea"],
        ["sobretensión sostenida","un cortocircuito monofásico","corriente de fuga elevada",
         "el disparo de la protección diferencial"],
        ["se aísla el circuito y se normaliza la tensión","se reemplaza el relé y se recalibra",
         "se reconecta el interruptor tras verificar la aislación"]),
     "falla_mecanica": (["el rodamiento del eje","la caja multiplicadora","el acoplamiento",
        "el sello del descanso","la bomba de lubricación"],
        ["vibración fuera de norma","temperatura elevada en el descanso","ruido anormal",
         "pérdida de aceite"],
        ["se reemplaza el rodamiento y se alinea el eje","se rellena y se purga el circuito de aceite",
         "se ajusta el acoplamiento y se mide la vibración"]),
     "evento_climatico": (["la línea de evacuación","el patio de alta tensión",
        "el camino de acceso","la estructura de la torre","el pararrayos del patio"],
        ["viento sobre lo previsto","una descarga atmosférica cercana","acumulación de nieve",
         "lluvia intensa con anegamiento"],
        ["se inspecciona la estructura y se despeja la faja","se repone el servicio al amainar",
         "se drena el sector y se revisa la puesta a tierra"]),
     "preventivo": (["el sistema de refrigeración","los filtros de aire","el tablero de control",
        "las conexiones de fuerza","el grupo hidráulico"],
        ["la mantención programada","el cambio de filtros","el ajuste de rutina",
         "la lubricación periódica"],
        ["se cambian filtros y se registra la lectura","se reaprietan las conexiones y se sella",
         "se completa la pauta sin observaciones"]),
     "correctivo": (["el equipo afectado","la unidad detenida","el componente dañado",
        "la sección fuera de servicio","el módulo de potencia"],
        ["la reparación de la falla del turno anterior","el levantamiento de la indisponibilidad",
         "la orden de trabajo pendiente","la intervención de emergencia"],
        ["se repara y se devuelve a servicio","se reemplaza la pieza y se prueba en vacío",
         "se normaliza y se informa al despacho"]),
     "inspeccion": (["el conjunto de medida","la señalética del área","los niveles de aceite",
        "el estado de los accesos","el registro de alarmas"],
        ["la ronda de rutina","la verificación visual","la lectura de instrumentos",
         "el chequeo de seguridad"],
        ["se deja constancia sin hallazgos","se anota una observación menor",
         "se programa revisión de detalle"]),
    }
    PLANT = ["Se registra {s} sobre {c}.",
             "Se detecta {s} en {c}, {a}.",
             "El operador reporta {s}, se revisa {c} y {a}.",
             "Evento por {s} en {c}, {a}."]
    def _obs(t):
        c, s, a = (VOCAB[t][k][int(rngm.integers(0, len(VOCAB[t][k])))] for k in (0, 1, 2))
        return PLANT[int(rngm.integers(0, len(PLANT)))].format(s=s, c=c, a=a)
    mant["observacion"] = [_obs(t) for t in mant["tipo_evento"]]

    # ---------------------------------------------------- el año 2025
    # El archivo nuevo del caso integrador. Tiene tres cosas adentro que el
    # participante va a tener que encontrar, y ademas llega con defectos de
    # formato porque viene de otra fuente.
    rng25 = np.random.default_rng(2025)
    fechas25 = pd.date_range("2025-01-01", "2025-12-31", freq="D")

    # 1. La demanda crece, pero no parejo. La Metropolitana crece poco y
    #    Antofagasta mucho, porque entro un consumo minero nuevo.
    CRECE = {"Metropolitana": 1.012, "Antofagasta": 1.094, "Atacama": 1.031,
             "Coquimbo": 1.025, "Valparaiso": 1.018, "Biobio": 1.021,
             "Los Lagos": 1.016}
    dem25 = []
    for r, p in zip(regs, pobl):
        base_r = p / 8000 * CRECE[r]
        for f in fechas25:
            inv = 1 + 0.18 * np.cos(2 * np.pi * (f.dayofyear - 190) / 365)
            finde = 0.92 if f.dayofweek >= 5 else 1.0
            v = base_r * perfil_d * inv * finde * rng25.uniform(.97, 1.03, 24)
            dem25.append(pd.DataFrame({"fecha": f.strftime("%Y-%m-%d"), "hora": range(24),
                                       "region": r, "mwh": v.round(2)}))
    demanda25 = pd.concat(dem25, ignore_index=True)

    # 2. La generación. Dos cosas cambian. El año fue seco, así que la hidro
    #    baja, y entró en servicio un parque solar nuevo a mitad de año.
    nueva = pd.DataFrame([("Parque Solar Rio Seco", "solar", "Atacama", 260, 2025)],
                         columns=["central", "tecnologia", "region", "potencia_mw", "anio_inicio"])
    centrales25 = pd.concat([centrales, nueva], ignore_index=True)
    filas25 = []
    for _, c in centrales25.iterrows():
        p, t = c.potencia_mw, c.tecnologia
        for f in fechas25:
            if c.central == "Parque Solar Rio Seco" and f < pd.Timestamp("2025-07-01"):
                base = np.zeros(24)                       # todavia no entraba en servicio
            else:
                est = 1 + 0.25 * np.cos(2 * np.pi * (f.dayofyear - 15) / 365)
                if t == "solar":    base = p * perfil * 0.30 * est * rng25.uniform(.8, 1.1)
                elif t == "eolica": base = p * 0.36 * rng25.uniform(.15, 1.6, 24)
                elif t == "hidro":  base = p * 0.55 * (2 - est) * rng25.uniform(.9, 1.1, 24) * 0.68
                elif t == "gas":    base = p * 0.68 * rng25.uniform(.9, 1.05, 24) * 1.12
                elif t == "carbon": base = p * 0.65 * rng25.uniform(.95, 1.02, 24)
                else:
                    base = np.zeros(24); base[18:23] = p * 0.55 * rng25.uniform(.8, 1, 5)
            filas25.append(pd.DataFrame({"fecha": f.strftime("%Y-%m-%d"), "hora": range(24),
                                         "central": c.central, "mwh": np.clip(base, 0, p).round(2)}))
    generacion25 = pd.concat(filas25, ignore_index=True)

    # 3. La bitácora del año, con una central que concentra fallas.
    ev25 = []
    n_dias25 = len(fechas25)
    for _, c in centrales25.iterrows():
        p = np.array(PESOS[c.tecnologia]); p = p / p.sum()
        n = 30 if c.tecnologia == "diesel" else 22
        if c.central == "Termoelectrica Puerto Sur":
            n = 64                                        # la central con problemas
            p = np.array([.10, .30, .24, .24, .02, .10]); p = p / p.sum()
        for tipo, d in zip(rng25.choice(TIPOS, size=n, p=p),
                           rng25.integers(0, n_dias25, size=n)):
            ev25.append([c.central, int(d), str(tipo)])
    mant25 = pd.DataFrame([{"central": c, "fecha": fechas25[d].strftime("%Y-%m-%d"),
                            "tipo_evento": t,
                            "duracion_horas": int(rng25.integers(DURA[t][0], DURA[t][1] + 1))}
                           for c, d, t in ev25])

    # El archivo llega con defectos de formato, porque viene de otra fuente.
    centrales25.to_csv("centrales_2025.csv", index=False)
    generacion25.to_csv("generacion_2025.csv", index=False)
    mant25.sort_values(["fecha", "central"], kind="stable").reset_index(drop=True) \
          .to_csv("mantenimiento_2025.csv", index=False)
    d25 = demanda25.copy()
    d25["fecha"] = pd.to_datetime(d25["fecha"]).dt.strftime("%d/%m/%Y")
    d25["mwh"] = d25["mwh"].map(lambda v: f"{v:.2f}".replace(".", ","))
    rr = rng25.choice(len(d25), size=int(len(d25) * 0.012), replace=False)
    d25.loc[rr, "region"] = d25.loc[rr, "region"].str.upper()
    d25.to_csv("demanda_2025.csv", sep=";", index=False)
    mant.sort_values(["fecha","central"], kind="stable").reset_index(drop=True) \
        .to_csv("mantenimiento.csv", index=False)
print("Datos listos. Los de 2024 y los de 2025, que son los del caso.")


In [ ]:
#@title Cómo se trabaja esta sesión { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Cómo se trabaja esta sesión</div><p>El notebook trae la estructura del trabajo y las preguntas. El código lo escribes tú.</p>
<p><strong>Trabaja en parejas</strong> si el relator lo indica. Discutir una decisión con alguien es la mitad del oficio.</p>
<p><strong>No busques la respuesta perfecta.</strong> Busca una respuesta defendible y deja escrito por qué la elegiste. Eso vale más que un número más exacto sin justificación.</p>
<p><strong>Los últimos veinte minutos son para el cierre.</strong> Reserva ese tiempo aunque no hayas terminado. Un informe incompleto y honesto sirve, uno completo e indefendible no.</p></div>"""))

## Parte 1. Abrir el archivo nuevo

*25 minutos. Módulos 1 y 3.*

El archivo `demanda_2025.csv` viene de otra fuente y llega con defectos. Los otros tres,
`generacion_2025.csv`, `centrales_2025.csv` y `mantenimiento_2025.csv`, llegan limpios.

Antes de calcular nada, diagnostica. Cuántas filas debería tener, de qué tipo es cada columna,
cuántos valores distintos hay donde debería haber pocos, y cuántos faltan.

In [ ]:
# 1.1 Mira el archivo tal como llega, sin arreglar nada.

import pandas as pd

# tu código acá

In [ ]:
# 1.2 El diagnóstico. Tipos, valores distintos y nulos.

# tu código acá

In [ ]:
# 1.3 Ábrelo bien, arreglando en la lectura lo que se puede arreglar ahí.

# tu código acá

In [ ]:
# 1.4 Y normaliza la región, que no se puede arreglar en la lectura.

# tu código acá

In [ ]:
# 1.5 Carga lo demás y deja todo listo para trabajar.

# tu código acá

In [ ]:
#@title Antes de seguir, anota { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Antes de seguir, anota</div><p>Escribe en una celda de texto qué defectos traía el archivo y qué hiciste con cada uno. Son cuatro líneas y es lo primero que va a preguntar quien revise el informe.</p></div>"""))

## Parte 2. Comparar 2025 con 2024

*25 minutos. Módulo 2.*

La primera pregunta del encargo. Cómo se comportó la demanda, y si el crecimiento fue parejo.

Ojo con una cosa. 2024 fue bisiesto y 2025 no, así que tienen distinto número de días.
Comparar los totales sin corregir eso es un error, y es exactamente el tipo de detalle que
alguien va a encontrar después.

In [ ]:
# 2.1 El total del sistema, corrigiendo por el número de días.

# tu código acá

In [ ]:
# 2.2 Y ahora por región, que es lo que de verdad pidieron.

# tu código acá

In [ ]:
# 2.3 La generación por tecnología, mismo tratamiento.

# tu código acá

In [ ]:
# 2.4 Hay una central que no estaba el año pasado.

# tu código acá

In [ ]:
#@title La pregunta que cierra la parte 2 { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">La pregunta que cierra la parte 2</div><p>La hidro cayó fuerte y el gas subió. ¿Son dos hechos o son uno solo? Escribe tu respuesta antes de seguir, y qué dato adicional pedirías para confirmarla.</p></div>"""))

## Parte 3. Qué se salió de la norma

*25 minutos. Módulos 4 y 5.*

La tercera pregunta del encargo. Si hay algo en la bitácora que valga la pena mirar.

Acá no hay una instrucción, hay una pregunta abierta. Tienes 512 eventos y veintiún
centrales. Encuentra lo que no calza.

In [ ]:
# 3.1 Lo más simple primero. Cuántos eventos tuvo cada central.

# tu código acá

In [ ]:
# 3.2 Esa central concentra eventos. De qué tipo son.

# tu código acá

In [ ]:
# 3.3 Compárala con el resto para saber si el número es raro de verdad.

# tu código acá

In [ ]:
# 3.4 Y la pregunta que importa. ¿Se le notó en la generación?

# tu código acá

In [ ]:
#@title Antes de seguir, decide { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Antes de seguir, decide</div><p>Con lo que tienes, ¿qué afirmarías en el informe sobre esa central y qué no? Escribe las dos listas. La segunda es tan importante como la primera.</p></div>"""))

## Parte 4. Las tres figuras del informe

*25 minutos. Módulo 6.*

El informe es de una página y lleva tres figuras, una por pregunta. Cada una tiene que
entenderse sola, con título, unidad y escala honesta.

In [ ]:
# 4.1 Figura 1. El crecimiento de la demanda por región.

import matplotlib.pyplot as plt

# tu código acá

In [ ]:
# 4.2 Figura 2. La generación por tecnología, los dos años.

import numpy as np

# tu código acá

In [ ]:
# 4.3 Figura 3. La central con problemas, mes a mes.

# tu código acá

In [ ]:
#@title La vuelta de revisión { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">La vuelta de revisión</div><p>Pásale a tus tres figuras la lista del Módulo 6. Título con periodo, ejes con unidad, orden si son categorías, escala desde cero en barras, y paleta que distinga cualquiera.</p>
<p>Casi seguro a alguna le falta algo.</p></div>"""))

## Parte 5. El cierre

*20 minutos. Este tramo no se salta.*

Un informe no es una lista de números, es un conjunto de afirmaciones que alguien va a
usar para decidir. Y cada afirmación tiene que poder rehacerse.

In [ ]:
# 5.1 Las tres afirmaciones del informe, armadas con los números calculados.
# Escribirlas a mano invita a que se desactualicen. Así no puede pasar.

# tu código acá

In [ ]:
#@title Lo que va en el informe y lo que no { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Lo que va en el informe y lo que no</div><p><strong>Va.</strong> Los tres hechos medidos, con su cifra, su periodo y la corrección por el número de días. Las tres figuras. Y la nota de que el archivo de demanda llegó con defectos de formato y qué se hizo con ellos.</p>
<p><strong>No va.</strong> La explicación de por qué cayó la hidro, porque no la tienes. Puedes decir que el patrón es compatible con un año seco y que hace falta el dato hidrológico para confirmarlo. Eso es honesto y además le dice a la jefatura qué pedir.</p>
<p><strong>Tampoco va.</strong> Que la central con eventos está fallando por antigüedad, o por mala operación, o por lo que sea. Lo que tienes es que concentra eventos. La causa la investiga quien corresponde, y tu trabajo es que sepan dónde mirar.</p></div>"""))

In [ ]:
#@title Puntos clave del curso { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d8f0df;border-left:6px solid #28a745;color:#14532d"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Puntos clave del curso</div><ul>
<li>Un archivo nuevo se diagnostica antes de calcularle nada, y los defectos se anotan</li>
<li>Comparar dos años exige corregir por el número de días, y ese detalle lo encuentra siempre alguien</li>
<li>El total del sistema esconde lo que pasa por región, y la pregunta casi nunca es el total</li>
<li>Un hallazgo es un lugar donde mirar, no una causa demostrada</li>
<li>Cada figura tiene que entenderse sola, y se revisa antes de mandarla</li>
<li>Lo que no puedes sostener se dice igual, diciendo qué dato haría falta</li>
</ul>
<p>Con esto termina el curso. Lo que se llevan no es una lista de funciones, es un método. Mirar antes de calcular, decidir a conciencia, medir lo que decidieron y dejarlo escrito.</p></div>"""))